# 5.5 LSTM、GRU、1D CNN 比較

這份 Notebook 在同一份時間序列分類資料上比較 LSTM、GRU 與 1D CNN 的表現、參數量與訓練時間。


## 1. 載入套件


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

tf.keras.utils.set_random_seed(42)


## 2. 建立共同資料集


In [ ]:
CLASS_NAMES = np.array(['trend', 'periodic', 'spike'])
TIMESTEPS = 48

def generate_sequence(label, rng, timesteps=TIMESTEPS):
    t = np.linspace(0, 1, timesteps)
    noise = rng.normal(0, 0.08, timesteps)
    if label == 0:
        value = 1.6 * t + 0.2 * np.sin(2 * np.pi * t) + noise
    elif label == 1:
        value = np.sin(2 * np.pi * 3 * t + rng.uniform(-0.4, 0.4)) + noise
    else:
        center = rng.integers(timesteps // 4, timesteps * 3 // 4)
        value = noise
        value += np.exp(-0.5 * ((np.arange(timesteps) - center) / 2.5) ** 2) * 2.0
    delta = np.diff(value, prepend=value[0])
    return np.stack([value, delta], axis=-1).astype('float32')

def make_sequence_classification(samples_per_class=260, seed=42):
    rng = np.random.default_rng(seed)
    x, y = [], []
    for label in range(len(CLASS_NAMES)):
        for _ in range(samples_per_class):
            x.append(generate_sequence(label, rng))
            y.append(label)
    x = np.stack(x).astype('float32')
    y = np.array(y, dtype='int64')
    idx = rng.permutation(len(y))
    return x[idx], y[idx]

def split_and_standardize(X, y):
    n = len(X)
    train_end = int(n * 0.7)
    valid_end = int(n * 0.85)
    x_train, y_train = X[:train_end], y[:train_end]
    x_valid, y_valid = X[train_end:valid_end], y[train_end:valid_end]
    x_test, y_test = X[valid_end:], y[valid_end:]
    mean = x_train.mean(axis=(0, 1), keepdims=True)
    std = x_train.std(axis=(0, 1), keepdims=True) + 1e-8
    return (x_train - mean) / std, y_train, (x_valid - mean) / std, y_valid, (x_test - mean) / std, y_test

X, y = make_sequence_classification(samples_per_class=230, seed=77)
x_train, y_train, x_valid, y_valid, x_test, y_test = split_and_standardize(X, y)
print(x_train.shape, x_valid.shape, x_test.shape)


## 3. 定義三種模型


In [ ]:
def build_lstm(input_shape, n_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dense(n_classes, activation='softmax'),
    ])
    return model

def build_gru(input_shape, n_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.GRU(32),
        tf.keras.layers.Dense(n_classes, activation='softmax'),
    ])
    return model

def build_conv1d(input_shape, n_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv1D(32, 5, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling1D(2),
        tf.keras.layers.Conv1D(64, 5, activation='relu', padding='same'),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(n_classes, activation='softmax'),
    ])
    return model


## 4. 訓練與比較


In [ ]:
def train_and_evaluate(name, builder):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)
    model = builder(x_train.shape[1:], len(CLASS_NAMES))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    start = time.perf_counter()
    history = model.fit(
        x_train, y_train,
        validation_data=(x_valid, y_valid),
        epochs=18,
        batch_size=32,
        verbose=0,
    )
    seconds = time.perf_counter() - start
    test_result = model.evaluate(x_test, y_test, verbose=0, return_dict=True)
    return {
        'model': name,
        'params': model.count_params(),
        'seconds': seconds,
        'best_val_accuracy': max(history.history['val_accuracy']),
        'test_accuracy': test_result['accuracy'],
        'test_loss': test_result['loss'],
    }

rows = [
    train_and_evaluate('LSTM', build_lstm),
    train_and_evaluate('GRU', build_gru),
    train_and_evaluate('1D CNN', build_conv1d),
]
results = pd.DataFrame(rows).sort_values('test_accuracy', ascending=False)
results


## 5. 視覺化比較結果


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
results.set_index('model')['test_accuracy'].plot(kind='bar', ax=axes[0], ylim=(0, 1.05), title='Test accuracy')
results.set_index('model')['params'].plot(kind='bar', ax=axes[1], title='Parameter count')
plt.tight_layout()
plt.show()


## 6. 如何套用自己的資料

用同一份資料切分、同一套前處理與同一指標比較模型。不要只比較最後 accuracy，也要看參數量、訓練時間、錯誤類型與部署限制。


## 7. 小結

LSTM、GRU、1D CNN 沒有絕對優劣。局部 pattern 明確時 1D CNN 通常很有競爭力；長期依賴明顯時，LSTM/GRU 會更自然。可靠的選型方式是建立可重跑的 baseline 比較。
